# Reproducing the results of the thesis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sara-almahri/supply-chain-disruption-monitoring/blob/main/notebooks/reproduce_results.ipynb)

This notebook recomputes the reported results of Chapters 3, 4 and 5 from the stored outputs of every run, checks that
the recomputed files equal the released ones, redraws the data figures, and recomputes the reference paths and risk
scores from the supply network. No language model is called, and no API key or Neo4j instance is needed. The whole
notebook takes about fifteen minutes on Colab; most of the time is spent on bootstrap intervals.

| Section | Thesis | Script |
|---|---|---|
| 1 | Chapter 3: Table 3.1, Section 3.4.2, Table 3.4 | `chapter3/score_ch3.py`, `chapter3/score_reruns.py` |
| 2 | Chapter 4: Tables 4.5 to 4.7 | `evaluation/benchmark_v2/score.py` |
| 3 | Chapter 5: Tables 5.3 to 5.8 | `scripts/score_v1_2.py`, `scripts/ch5_analysis_v1_2.py` |
| 4 | Recomputed files against the released files | |
| 5 | Figures 3.7, 3.8, 5.4, 5.5 and the case-study figures of Section 4.5 | `chapter3/figures_ch3.py`, `scripts/figures_ch5_v1_2.py`, `thesis_figures/figures_ch4_case_study.py` |
| 6 | References of Chapters 4 and 5 recomputed from the network | `offline/check_references.py` |
| 7 | Unit tests | `analysis/tests`, `offline/tests` |

Metric definitions are in [`analysis/METRICS.md`](https://github.com/sara-almahri/supply-chain-disruption-monitoring/blob/main/analysis/METRICS.md).

In [ ]:
import os
if not os.path.exists("supply-chain-disruption-monitoring"):
    !git clone --depth 1 https://github.com/sara-almahri/supply-chain-disruption-monitoring.git
%cd supply-chain-disruption-monitoring
!pip install -q -r requirements.txt

## 1. Chapter 3: entity and relation extraction

The ten stored GPT-4o runs on the evaluation documents, scored against the reference annotation
(`chapter3/reference_final`): precision, recall, F1 and MCC per type and overall, with 95% bootstrap intervals over
documents and a chance baseline for every type (Table 3.1). The second script scores the prompt variants and the other
models (Section 3.4.2 and Table 3.4).

In [ ]:
!python chapter3/score_ch3.py
!python chapter3/score_reruns.py

## 2. Chapter 4: relevance benchmark

The 1,117 items of the test set (120 positives, 997 negatives): detection by the pipeline and by the comparison, with
95% bootstrap intervals and the always-alert baseline (Table 4.5), the agent decisions on the positives (Table 4.6) and
precision at lower prevalence (Table 4.7).

In [ ]:
!python -m evaluation.benchmark_v2.score

## 3. Chapter 5: multi-level evaluation

The 630 runs on the 30 scenarios. The first script scores every run against its scenario reference and gives the agent
decisions (Table 5.4); the second gives each agent's correctness on the input it received, error propagation, decision
correctness, constrained success, pass^k and the paired differences of the input variations. Together they give
Tables 5.3 to 5.8.

In [ ]:
!python scripts/score_v1_2.py
!python scripts/ch5_analysis_v1_2.py

## 4. Recomputed files against the released files

Every file written above is compared with the released version. Numbers are compared to a relative tolerance of
10<sup>-9</sup>, because another Python or NumPy version can change the last floating-point digit of a sum.

In [ ]:
import csv, io, json, math, subprocess

FILES = ["chapter3/results/ch3_scores.json", "chapter3/results/reruns_scores.json",
         "evaluation/results/benchmark_v2/test/scores.json", "evaluation/results/analysis/v1_2_full_scores.json",
         "evaluation/results/analysis/v1_2_full_runs.csv", "evaluation/results/analysis/v1_2_ch5_analysis.json"]

def number(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def differences(a, b, path=""):
    if isinstance(a, dict) and isinstance(b, dict):
        if a.keys() != b.keys():
            return [path + ": different keys"]
        return [d for k in a for d in differences(a[k], b[k], f"{path}/{k}")]
    if isinstance(a, list) and isinstance(b, list):
        if len(a) != len(b):
            return [path + ": different lengths"]
        return [d for i, (x, y) in enumerate(zip(a, b)) for d in differences(x, y, f"{path}[{i}]")]
    x, y = number(a), number(b)
    if x is not None and y is not None and not isinstance(a, bool):
        same = (math.isnan(x) and math.isnan(y)) or math.isclose(x, y, rel_tol=1e-9, abs_tol=1e-12)
    else:
        same = a == b
    return [] if same else [f"{path}: {a!r} != {b!r}"]

def load(text, name):
    return list(csv.reader(io.StringIO(text))) if name.endswith(".csv") else json.loads(text)

for f in FILES:
    released = subprocess.run(["git", "show", f"HEAD:{f}"], capture_output=True, text=True, check=True).stdout
    found = differences(load(open(f).read(), f), load(released, f))
    print(("identical  " if not found else f"{len(found)} differences  ") + f)
    for d in found[:5]:
        print("    ", d)

## 5. Figures

The data figures, drawn from the files above.

In [ ]:
!python chapter3/figures_ch3.py
!python scripts/figures_ch5_v1_2.py
!python thesis_figures/figures_ch4_case_study.py

In [ ]:
from IPython.display import Image, display

for png in ["chapter3/figures/ground-truth-entities.png", "chapter3/figures/extracted-entities.png",
            "thesis_figures/out/ch5/repeatability.png", "thesis_figures/out/ch5/input-variation.png",
            "thesis_figures/out/ch4/casestudy-tier2-paths.png", "thesis_figures/out/ch4/casestudy-risk-manager.png"]:
    print(png)
    display(Image(png, width=700))

## 6. References recomputed from the supply network

The label, the disrupted paths of each tier, the Tier-1 risk scores (Equation 4.1) and the expected CSCO actions of
every scenario of Chapter 5 and every benchmark item of Chapter 4 are recomputed from their specifications on the
released network (`dataset/supplychainKG_clean.csv`) with NetworkX, and compared with the released references. About
five minutes.

In [ ]:
!python -m offline.check_references --set all

## 7. Unit tests

In [ ]:
!python -m unittest discover -s analysis/tests -t .
!python -m unittest discover -s offline/tests -t .

## Running the pipeline

Running the agents themselves needs an OpenAI API key and a Neo4j instance loaded with the released network; see the
[README](https://github.com/sara-almahri/supply-chain-disruption-monitoring#running-the-pipeline).